# Projeto de Visualização Computacional (SCC5836)
## Etapa 2: Análise Visual e Discussões Preliminares
**ICMC - Universidade de São Paulo (USP)**

**Professor:** Jean R. Ponciano

**Grupo:**
- Camila Donda Ronchi
- Henrique Gualberto Marques
- Sandra Bastos de Almeida

Este notebook continua o `01_pre_processamento.ipynb`. Ele parte da base já tratada (`disciplinas_limpo.csv`) e traz a apresentação do problema, as questões de investigação, as visualizações interativas e as discussões preliminares.

> **Execução:** o notebook roda no Google Colab e localmente. No Colab, se o `disciplinas_limpo.csv` não estiver na pasta de arquivos, a célula de carregamento abre um botão de upload. Use "Executar tudo" e aguarde.


## Apresentação

### Motivação e contextualização
Quem faz pós-graduação na USP escolhe disciplinas pelo Janus, que mostra uma disciplina por vez, em formato de ficha. É difícil enxergar o conjunto: quantas disciplinas cada programa oferece, quanto tempo se espera que o aluno estude fora da sala, em que campus e em qual formato elas acontecem.

Nosso conjunto de dados são as disciplinas de pós-graduação cadastradas no Janus, coletadas com um crawler (Scrapy) e tratadas no notebook anterior. O dia a dia universitário que ele retrata é o nosso: montar a grade de um semestre, comparar carga de trabalho entre programas e entender como a pós está organizada entre os campi.

O interesse do problema vai além de listar disciplinas. A USP organiza a pós em uma hierarquia (campus, comissão, programa, área de concentração), mas essa estrutura administrativa não garante que disciplinas do mesmo programa se pareçam, nem que programas de áreas diferentes tenham o mesmo perfil de carga horária. As visualizações abaixo tentam olhar para isso.

### 0. Configuração do ambiente

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Constante para identificar ambiente (Google Colab ou Local)
IN_COLAB = 'google.colab' in sys.modules

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 1000)
pio.templates.default = 'plotly_white'

In [2]:
# Procurando o arquivo da base tratada em alguns caminhos comuns
caminhos = [
    Path('disciplinas_limpo.csv'),
    Path('data/disciplinas_limpo.csv'),
    Path('../data/disciplinas_limpo.csv'),
    Path('/content/disciplinas_limpo.csv'),
    Path('/content/data/disciplinas_limpo.csv'),
]
caminho_csv = next((c for c in caminhos if c.exists()), None)

# No Colab, se não achou, pede o upload do arquivo
if caminho_csv is None and IN_COLAB:
    from google.colab import files
    print('Arquivo não encontrado. Selecione o disciplinas_limpo.csv para enviar:')
    enviado = files.upload()
    caminho_csv = Path(next(iter(enviado)))

if caminho_csv is None:
    raise FileNotFoundError('disciplinas_limpo.csv não encontrado. Coloque o arquivo na pasta do notebook ou em data/.')

df = pd.read_csv(caminho_csv)
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Arquivo: {caminho_csv}')
print(f'Dimensão: {df.shape[0]} disciplinas e {df.shape[1]} colunas')

Ambiente: Google Colab
Arquivo: disciplinas_limpo.csv
Dimensão: 2538 disciplinas e 34 colunas


In [3]:
# Paleta de cores segura para daltonismo (Okabe-Ito), uma cor por grande área
CORES_AREA = {
    'Ciências Agrárias': '#E69F00',
    'Ciências Biológicas e da Saúde': '#009E73',
    'Ciências Exatas e da Terra': '#56B4E9',
    'Engenharias': '#0072B2',
    'Humanidades e Sociais Aplicadas': '#D55E00',
    'Multidisciplinar': '#CC79A7',
}
AREAS = list(CORES_AREA)

# Rótulos curtos para caber nos eixos
ROTULO_AREA = {
    'Ciências Agrárias': 'Agrárias',
    'Ciências Biológicas e da Saúde': 'Biológicas e Saúde',
    'Ciências Exatas e da Terra': 'Exatas e da Terra',
    'Engenharias': 'Engenharias',
    'Humanidades e Sociais Aplicadas': 'Humanas e Sociais',
    'Multidisciplinar': 'Multidisciplinar',
}

CORES_CAMPUS = {
    'São Paulo': '#0072B2',
    'Ribeirão Preto': '#D55E00',
    'São Carlos': '#009E73',
    'Piracicaba': '#E69F00',
    'Bauru': '#CC79A7',
    'Lorena': '#56B4E9',
    'Pirassununga': '#000000',
    'Intercampi / Interinstitucional': '#999999',
}

CINZA = '#BDBDBD'
FONTE = dict(family='Arial, sans-serif', size=13)

def ajustar(fig, titulo, altura=600):
    fig.update_layout(title=dict(text=titulo, x=0.01), font=FONTE, height=altura,
                      margin=dict(l=60, r=40, t=120, b=60))
    return fig

## Introdução

### Informações sobre o conjunto de dados
A base tratada tem uma linha por disciplina de pós-graduação cadastrada no Janus. A tabela abaixo organiza as 34 colunas por tipo de informação, com o tipo no pandas, o número de valores distintos e a quantidade de nulos.

In [4]:
grupos = {
    'Identificação': ['codigo', 'disciplina'],
    'Estrutura institucional': ['campus', 'grande_area', 'comissao', 'codigo_comissao', 'nome_programa',
                                'codigo_programa', 'area_concentracao', 'codigo_area_concentracao'],
    'Carga horária (numéricas)': ['n_creditos', 'carga_total_horas', 'duracao_semanas', 'carga_teorica',
                                  'carga_pratica', 'carga_estudo', 'carga_aula_semanal', 'carga_semanal_total'],
    'Proporções derivadas': ['prop_teorica', 'prop_pratica', 'prop_estudo', 'razao_estudo_aula'],
    'Formato da disciplina': ['tipo_duracao', 'is_intensiva', 'idioma_cat', 'oferecimento_cat'],
    'Docentes e data': ['num_docentes', 'docentes_responsaveis', 'ano_criacao'],
    'Texto livre': ['objetivos', 'justificativa', 'conteudo', 'avaliacao', 'bibliografia'],
}

linhas = []
for grupo, colunas in grupos.items():
    for c in colunas:
        linhas.append({'grupo': grupo, 'variável': c, 'tipo': str(df[c].dtype),
                       'valores distintos': df[c].nunique(), 'nulos': int(df[c].isna().sum())})

resumo = pd.DataFrame(linhas)
display(resumo)
print(f'Colunas descritas: {len(resumo)} de {df.shape[1]}')

,grupo,variável,tipo,valores distintos,nulos
0,Identificação,codigo,object,2538,0
1,Identificação,disciplina,object,2495,0
2,Estrutura institucional,campus,object,8,0
3,Estrutura institucional,grande_area,object,6,0
4,Estrutura institucional,comissao,object,64,0
5,Estrutura institucional,codigo_comissao,int64,64,0
6,Estrutura institucional,nome_programa,object,247,0
7,Estrutura institucional,codigo_programa,int64,270,0
8,Estrutura institucional,area_concentracao,object,339,0
9,Estrutura institucional,codigo_area_concentracao,int64,362,0


Colunas descritas: 34 de 34


Em termos de tipo de variável, temos:

- **Categóricas nominais:** campus, grande área, comissão, programa, área de concentração, idioma e modalidade (presencial ou não). Algumas têm hierarquia (campus > área > comissão > programa > área de concentração).
- **Quantitativas discretas:** créditos, carga horária semanal (teórica, prática e de estudo), duração em semanas, carga total e número de docentes.
- **Quantitativas contínuas (derivadas):** proporções teórica, prática e de estudo, e a razão estudo/aula.
- **Textuais:** objetivos, justificativa, conteúdo, avaliação e bibliografia.

Como a USP define 1 crédito = 15 horas, `n_creditos` é igual a `carga_total_horas / 15` e carrega a mesma informação que a carga total (validado no notebook 01). Por isso tratamos as duas como uma só informação e não usamos as duas como variáveis separadas.

### Questões que queremos explorar
1. **Perfil de carga horária.** Como teoria, prática e estudo autônomo se distribuem entre as grandes áreas? Há áreas mais práticas, ou que exigem mais estudo fora da sala?
2. **Estrutura da oferta.** Como as disciplinas se distribuem na hierarquia campus, área e programa? Onde a oferta se concentra?
3. **Formato.** Como disciplinas intensivas, não presenciais e em outro idioma se distribuem entre campi e áreas?
4. **Casos extremos.** Quais disciplinas fogem do padrão de carga (semanal total ou razão estudo/aula)? Existe alguma explicação estrutural para elas?
5. **Similaridade de conteúdo.** Ao olhar só o texto (objetivos e conteúdo), disciplinas da mesma área ficam próximas? Quais ficam "entre" áreas?

## Pré-processamento

A limpeza principal está no `01_pre_processamento.ipynb`, e aqui só retomamos o que importa para a análise. Naquele notebook descartamos as disciplinas sem créditos e carga horária no Janus, convertemos os tipos (semanas, horas e data de criação), confirmamos as regras da USP (carga total = soma das cargas semanais x semanas, e 15 horas por crédito), criamos as proporções e a razão estudo/aula, marcamos como intensivas as disciplinas de até 4 semanas, categorizamos idioma e modalidade e, por fim, mapeamos cada comissão para campus e grande área.

Neste notebook foram necessários mais dois ajustes, descritos abaixo.

### 1. Docentes ausentes não são "zero docentes"
No notebook 01 o campo `num_docentes` vale 0 quando `docentes_responsaveis` está vazio. Isso mistura "sem docente" com "docente não informado no Janus". Verificamos o tamanho do problema abaixo. Como a informação de docentes tem muitas lacunas, não a usamos nas visualizações desta entrega.

In [5]:
# Tratando num_docentes = 0 como dado ausente
df['num_docentes_aj'] = df['num_docentes'].where(df['num_docentes'] > 0)

ausentes = df['num_docentes_aj'].isna()
print(f'Disciplinas sem docentes informados: {ausentes.sum()} ({ausentes.mean():.1%})')

display(
    df.groupby('grande_area')['num_docentes_aj']
    .apply(lambda s: s.isna().mean())
    .mul(100).round(1)
    .rename('% sem docente informado')
    .to_frame()
)

Disciplinas sem docentes informados: 1018 (40.1%)


,% sem docente informado
grande_area,
Ciências Agrárias,50.0
Ciências Biológicas e da Saúde,25.4
Ciências Exatas e da Terra,32.2
Engenharias,48.7
Humanidades e Sociais Aplicadas,50.3
Multidisciplinar,36.0


### 2. Vetorização do texto
Para a questão 5, `disciplina`, `objetivos` e `conteudo` são transformados em vetores TF-IDF (com redução por SVD). Esse passo fica na seção 6, junto da visualização, porque os parâmetros são parte da análise.

## Análise visual

Todas as visualizações usam Plotly e são interativas (tooltip, zoom, legenda clicável, seletores e filtros). As cores das grandes áreas seguem a paleta Okabe-Ito, pensada para daltonismo. Sempre que a cor é essencial para separar grupos, a informação também aparece por outro canal (rótulo, posição ou número na célula).

| Visualização | Tipo | Questão |
|---|---|---|
| Treemap | Árvore / hierarquia | 2 |
| Boxplot com seletor de métrica | Distribuição | 1 e 4 |
| Heatmap com seletor de métrica | Matriz categórica | 3 |
| Projeção t-SNE do texto | Redução de dimensionalidade | 5 |

### 1. Estrutura da oferta: treemap (questão 2)

**Por que este gráfico.** A organização da pós é uma hierarquia, e a pergunta é onde a oferta se concentra. O treemap mostra o todo e as partes ao mesmo tempo, e a área de cada retângulo é proporcional ao número de disciplinas. Um gráfico de barras com 247 programas ficaria ilegível. Aqui é possível clicar em um retângulo para entrar no nível seguinte, e a barra no topo permite voltar. O menu alterna a ordem da hierarquia (campus > área ou área > campus). Campus aparece em cinza e as áreas e programas usam a cor da grande área.

In [6]:
def montar_hierarquia(dados, niveis):
    # Monta ids, rótulos, pais, valores e cores para um treemap com vários níveis
    ids, rotulos, pais, valores, cores = [], [], [], [], []
    for i in range(len(niveis)):
        cols = niveis[:i + 1]
        cont = dados.groupby(cols).size().reset_index(name='n')
        for _, linha in cont.iterrows():
            caminho = [str(linha[c]) for c in cols]
            ids.append(' | '.join(caminho))
            pais.append(' | '.join(caminho[:-1]))
            valores.append(int(linha['n']))
            rotulo = caminho[-1]
            if cols[-1] == 'grande_area':
                rotulo = ROTULO_AREA[rotulo]
            rotulos.append(rotulo)
            cores.append(CORES_AREA[linha['grande_area']] if 'grande_area' in cols else CINZA)
    return ids, rotulos, pais, valores, cores


HIERARQUIAS = {
    'Campus > Área > Programa': ['campus', 'grande_area', 'nome_programa'],
    'Área > Campus > Programa': ['grande_area', 'campus', 'nome_programa'],
}
primeira = list(HIERARQUIAS)[0]

fig = go.Figure()
for nome, niveis in HIERARQUIAS.items():
    ids, rotulos, pais, valores, cores = montar_hierarquia(df, niveis)
    fig.add_trace(go.Treemap(
        ids=ids, labels=rotulos, parents=pais, values=valores,
        branchvalues='total',
        marker=dict(colors=cores, line=dict(width=1.5, color='white')),
        textinfo='label+value',
        hovertemplate='<b>%{label}</b><br>%{value} disciplinas<br>%{percentRoot:.1%} do total<extra></extra>',
        pathbar=dict(visible=True),
        visible=(nome == primeira),
    ))

botoes = []
for i, nome in enumerate(HIERARQUIAS):
    visivel = [j == i for j in range(len(HIERARQUIAS))]
    botoes.append(dict(label=nome, method='update', args=[{'visible': visivel}]))

fig.update_layout(updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, xanchor='left',
                                    y=1.0, yanchor='bottom', showactive=True)])
ajustar(fig, 'Disciplinas de pós-graduação por campus, área e programa (clique para aprofundar)', 650)
fig.show()

In [7]:
# Tabela de apoio: concentração por campus e por programa
por_campus = df.groupby('campus').agg(disciplinas=('codigo', 'size'),
                                      programas=('nome_programa', 'nunique')).sort_values('disciplinas', ascending=False)
por_campus['% das disciplinas'] = (por_campus['disciplinas'] / len(df) * 100).round(1)
display(por_campus)

por_programa = df['nome_programa'].value_counts()
print(f'Total de programas: {por_programa.shape[0]}')
print(f'Os 10 maiores programas somam {por_programa.head(10).sum()} disciplinas '
      f'({por_programa.head(10).sum() / len(df):.1%} do total)')
display(por_programa.head(10).rename('disciplinas').to_frame())

,disciplinas,programas,% das disciplinas
campus,,,
São Paulo,1434,159,56.5
Ribeirão Preto,418,52,16.5
São Carlos,272,17,10.7
Piracicaba,160,15,6.3
Intercampi / Interinstitucional,103,14,4.1
Lorena,62,6,2.4
Bauru,56,3,2.2
Pirassununga,33,4,1.3


Total de programas: 247
Os 10 maiores programas somam 539 disciplinas (21.2% do total)


,disciplinas
nome_programa,
Engenharia Elétrica,87
Direito,75
Física,65
Engenharia Mecânica,58
Química,50
Tecnologia Nuclear,43
Música,42
Matemática,41
Saúde Pública,40


**O que vemos.** São Paulo ocupa mais da metade do gráfico: são 1.434 das 2.538 disciplinas (56%) e 159 dos 247 programas. Ribeirão Preto vem em seguida, com 418, e nenhum dos outros campi passa de 272. Dentro de São Paulo, Humanas e Sociais é a maior área (605), seguida de Biológicas e Saúde (352).

Trocando para a hierarquia Área > Campus, vemos que Engenharias se divide basicamente entre São Paulo (191), São Carlos (169) e Lorena (62), que Biológicas e Saúde tem muito peso em Ribeirão Preto (266) e que Agrárias está mais em Piracicaba (160 de 260). A concentração também aparece nos programas: Engenharia Elétrica (87), Direito (75) e Física (65) lideram, e os 10 maiores programas ficam com 21% das disciplinas, apesar de serem só 4% dos 247 programas.

### 2. Perfil de carga horária por área: boxplot com seletor de métrica (questões 1 e 4)

**Por que este gráfico.** Queremos comparar distribuições, não só médias, entre seis grupos. O boxplot mostra mediana, quartis e outliers, e a linha tracejada marca a média. O menu troca a métrica sem trocar de figura (teoria, prática, estudo, carga semanal total, razão estudo/aula e duração), o que permite comparar as componentes da carga. Os pontos fora da caixa são as disciplinas atípicas, e o tooltip mostra o nome, o programa, a duração e a carga total de cada uma. A duração é importante para interpretar os outliers de carga semanal.

In [8]:
METRICAS = {
    'Carga teórica (h/semana)': 'carga_teorica',
    'Carga prática (h/semana)': 'carga_pratica',
    'Estudo autônomo (h/semana)': 'carga_estudo',
    'Carga semanal total (h)': 'carga_semanal_total',
    'Razão estudo / aula': 'razao_estudo_aula',
    'Duração (semanas)': 'duracao_semanas',
}

fig = go.Figure()
for i, (rotulo, col) in enumerate(METRICAS.items()):
    for area in AREAS:
        sub = df[df['grande_area'] == area]
        fig.add_trace(go.Box(
            y=sub[col], name=ROTULO_AREA[area],
            marker_color=CORES_AREA[area],
            boxpoints='outliers', boxmean=True,
            hovertext=(sub['disciplina'] + '<br>' + sub['nome_programa'] + '<br>'
                       + sub['duracao_semanas'].astype(str) + ' semanas · '
                       + sub['carga_total_horas'].astype(str) + ' h no total').tolist(),
            visible=(i == 0), showlegend=False,
        ))

total_tracos = len(METRICAS) * len(AREAS)
botoes = []
for i, rotulo in enumerate(METRICAS):
    visivel = [j // len(AREAS) == i for j in range(total_tracos)]
    botoes.append(dict(label=rotulo, method='update',
                       args=[{'visible': visivel}, {'yaxis.title.text': rotulo}]))

fig.update_layout(updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, xanchor='left',
                                    y=1.0, yanchor='bottom', showactive=True)],
                  yaxis_title=list(METRICAS)[0])
ajustar(fig, 'Distribuição das métricas de carga horária por grande área', 600)
fig.show()

In [9]:
# Tabela de apoio: medianas por área
cols_resumo = ['carga_teorica', 'carga_pratica', 'carga_estudo', 'carga_semanal_total',
               'razao_estudo_aula', 'duracao_semanas']
med = df.groupby('grande_area')[cols_resumo].median().round(2)
med['prop_pratica (média)'] = df.groupby('grande_area')['prop_pratica'].mean().round(2)
med['n'] = df.groupby('grande_area').size()
display(med)

# Médias de estudo e prática, que mostram melhor a diferença quando a mediana empata
display(df.groupby('grande_area')[['carga_pratica', 'carga_estudo']].mean().round(2))

,carga_teorica,carga_pratica,carga_estudo,carga_semanal_total,razao_estudo_aula,duracao_semanas,prop_pratica (média),n
grande_area,,,,,,,,
Ciências Agrárias,4.0,2.0,4.0,8.0,0.50,11.0,0.33,260
Ciências Biológicas e da Saúde,3.0,3.0,2.0,10.0,0.50,9.0,0.49,681
Ciências Exatas e da Terra,4.0,1.0,4.0,10.0,1.00,15.0,0.25,311
Engenharias,3.0,1.0,7.0,10.0,1.50,12.0,0.24,427
Humanidades e Sociais Aplicadas,4.0,2.0,2.0,10.0,0.43,12.0,0.39,684
Multidisciplinar,4.0,1.0,4.0,9.0,0.67,10.0,0.32,175


,carga_pratica,carga_estudo
grande_area,,
Ciências Agrárias,3.01,3.77
Ciências Biológicas e da Saúde,4.07,3.85
Ciências Exatas e da Terra,1.65,5.14
Engenharias,1.35,6.38
Humanidades e Sociais Aplicadas,2.59,3.24
Multidisciplinar,2.41,3.99


In [ ]:
# Tabela de apoio (questão 4): carga semanal por duração da disciplina
por_duracao = df.groupby('duracao_semanas').agg(
    disciplinas=('codigo', 'size'),
    carga_semanal_media=('carga_semanal_total', 'mean'),
    carga_total_mediana=('carga_total_horas', 'median'),
).round(1)
display(por_duracao)

# Disciplinas com maior carga semanal total
display(
    df.sort_values('carga_semanal_total', ascending=False)
    [['disciplina', 'nome_programa', 'campus', 'carga_teorica', 'carga_pratica',
      'carga_estudo', 'duracao_semanas', 'carga_total_horas']].head(8)
)

# Casos com muito mais estudo do que aula e disciplinas sem algum tipo de carga
print(f'Razão estudo/aula > 2: {(df["razao_estudo_aula"] > 2).sum()} disciplinas '
      f'({(df["razao_estudo_aula"] > 2).mean():.1%})')
print(f'Sem estudo autônomo: {(df["carga_estudo"] == 0).sum()} | '
      f'sem prática: {(df["carga_pratica"] == 0).sum()} | sem teoria: {(df["carga_teorica"] == 0).sum()}')

**O que vemos.** A carga semanal total é parecida entre as áreas, com mediana de 8 a 10 horas. O que muda é a composição. Em Engenharias a mediana de estudo autônomo é de 7 h/semana e a prática é de 1 h, enquanto em Biológicas e Saúde e em Humanas e Sociais o estudo mediano é de 2 h. Biológicas e Saúde é a área mais prática (mediana de 3 h/semana e proporção prática média de 0,49), e Engenharias e Exatas ficam em 0,24 e 0,25. A duração mediana também varia: 15 semanas em Exatas e 9 em Biológicas e Saúde.

Na razão estudo/aula, os outliers chegam a valores bem altos (a maior é 9, com 2 h de aula e 18 h de estudo), e há muitos pontos acima da caixa em quase todas as áreas. A razão estudo/aula alta (166 disciplinas, 6,5%) vem de poucas horas de aula e muito estudo, o que aparece bastante em programas como Psicologia Clínica e algumas Engenharias. Além disso, 132 disciplinas não têm carga de estudo (5,2%) e 565 não têm prática (22,3%).

Os outliers da carga semanal total têm outra origem. Passando o mouse nos pontos mais altos, o tooltip mostra duração de 1 ou 2 semanas: são disciplinas de 30 a 60 horas comprimidas em pouco tempo, com carga semanal de até 30 h (por exemplo, 10 h de teoria, 10 de prática e 10 de estudo em uma semana). A tabela de apoio confirma: as disciplinas de 1 semana têm carga semanal média de 28 h, e as de 15 semanas, 8 h. Ou seja, esse tipo de outlier é efeito da duração, e não de disciplinas "pesadas".

### 3. Formato das disciplinas por campus e área: heatmap com seletor (questão 3)

**Por que este gráfico.** Aqui cruzamos duas variáveis categóricas (campus e grande área) e queremos comparar um valor em cada combinação. O heatmap dispensa leitura de eixo contínuo e deixa o número escrito na célula, então a cor não é o único canal. O menu muda a medida (contagem, % de intensivas, % não presenciais, % em outro idioma, proporção prática e carga semanal média). Células cinzas são combinações sem nenhuma disciplina. O tooltip mostra o `n` da célula, importante porque percentuais de células pequenas oscilam muito.

In [12]:
dh = df.copy()
dh['pct_intensiva'] = dh['is_intensiva'].astype(float) * 100
dh['pct_nao_presencial'] = (dh['oferecimento_cat'] == 'Não-Presencial').astype(float) * 100
dh['pct_outro_idioma'] = (dh['idioma_cat'] != 'Português').astype(float) * 100
dh['pct_pratica'] = dh['prop_pratica'] * 100

ordem_campus = dh['campus'].value_counts().index.tolist()
rotulos_x = [ROTULO_AREA[a] for a in AREAS]

# (rótulo do menu, coluna, agregação, formato do texto na célula)
MEDIDAS = [
    ('Nº de disciplinas', 'codigo', 'count', '%{z:.0f}'),
    ('% intensivas (até 4 semanas)', 'pct_intensiva', 'mean', '%{z:.0f}%'),
    ('% não presenciais', 'pct_nao_presencial', 'mean', '%{z:.0f}%'),
    ('% em outro idioma (não português)', 'pct_outro_idioma', 'mean', '%{z:.0f}%'),
    ('Proporção prática média (%)', 'pct_pratica', 'mean', '%{z:.0f}%'),
    ('Carga semanal total média (h)', 'carga_semanal_total', 'mean', '%{z:.1f}'),
]

def matriz(coluna, agregacao):
    m = dh.pivot_table(index='campus', columns='grande_area', values=coluna, aggfunc=agregacao)
    return m.reindex(index=ordem_campus, columns=AREAS)

n_celula = matriz('codigo', 'count').fillna(0).astype(int)

fig = go.Figure()
for i, (rotulo, coluna, agregacao, formato) in enumerate(MEDIDAS):
    z = matriz(coluna, agregacao)
    fig.add_trace(go.Heatmap(
        z=z.values, x=rotulos_x, y=ordem_campus,
        customdata=n_celula.values,
        colorscale='Viridis', xgap=2, ygap=2, hoverongaps=False,
        texttemplate=formato,
        colorbar=dict(title=dict(text=rotulo, side='right')),
        hovertemplate='<b>%{y}</b> · %{x}<br>' + rotulo + ': %{z:.1f}<br>n = %{customdata} disciplinas<extra></extra>',
        visible=(i == 0),
    ))

botoes = []
for i, medida in enumerate(MEDIDAS):
    visivel = [j == i for j in range(len(MEDIDAS))]
    botoes.append(dict(label=medida[0], method='update', args=[{'visible': visivel}]))

fig.update_layout(updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, xanchor='left',
                                    y=1.0, yanchor='bottom', showactive=True)],
                  yaxis=dict(autorange='reversed'), plot_bgcolor='#E5E5E5')
ajustar(fig, 'Formato das disciplinas por campus e grande área', 560)
fig.show()

In [13]:
# Tabelas de apoio: formato por área e por campus
def formato_por(coluna):
    t = dh.groupby(coluna).agg(
        n=('codigo', 'size'),
        pct_intensiva=('pct_intensiva', 'mean'),
        pct_nao_presencial=('pct_nao_presencial', 'mean'),
        pct_outro_idioma=('pct_outro_idioma', 'mean'),
    ).round(1)
    return t.sort_values('n', ascending=False)

display(formato_por('grande_area'))
display(formato_por('campus'))
print(f'Disciplinas fora do português: {(df["idioma_cat"] != "Português").sum()} '
      f'({(df["idioma_cat"] != "Português").mean():.1%})')

,n,pct_intensiva,pct_nao_presencial,pct_outro_idioma
grande_area,,,,
Humanidades e Sociais Aplicadas,684,7.2,18.7,4.2
Ciências Biológicas e da Saúde,681,27.9,21.3,7.5
Engenharias,427,3.3,11.7,5.2
Ciências Exatas e da Terra,311,7.7,6.8,11.9
Ciências Agrárias,260,23.5,14.6,4.6
Multidisciplinar,175,20.0,15.4,4.0


,n,pct_intensiva,pct_nao_presencial,pct_outro_idioma
campus,,,,
São Paulo,1434,16.4,17.8,7.1
Ribeirão Preto,418,19.1,13.2,6.5
São Carlos,272,4.8,11.4,5.9
Piracicaba,160,7.5,6.2,2.5
Intercampi / Interinstitucional,103,11.7,33.0,2.9
Lorena,62,0.0,8.1,1.6
Bauru,56,17.9,19.6,3.6
Pirassununga,33,33.3,24.2,9.1


Disciplinas fora do português: 158 (6.2%)


**O que vemos.** Disciplinas intensivas (até 4 semanas) são mais comuns em Biológicas e Saúde (28%), Agrárias (24%) e Multidisciplinar (20%), e raras em Engenharias (3%), Exatas (8%) e Humanas e Sociais (7%). Por campus, Pirassununga tem 33% de intensivas e Ribeirão Preto 19%, enquanto Lorena não tem nenhuma e São Carlos tem 5%. Disciplinas não presenciais são 33% no conjunto Intercampi/Interinstitucional, 24% em Pirassununga e 20% em Bauru, contra 6% em Piracicaba e 8% em Lorena. Disciplinas fora do português são poucas (6,2% do total), mas se concentram em Exatas (12%).

O heatmap mostra também que campus e área não são independentes: Lorena só tem Engenharias, Bauru só tem Biológicas e Saúde e Piracicaba só tem Agrárias. Isso vem do mapeamento comissão para campus e área do notebook 01, então comparar campi muitas vezes é comparar áreas. Por isso olhamos as células com cuidado: em células com poucas disciplinas (como Engenharias no Intercampi, com 5) os percentuais não são confiáveis.

### 4. Similaridade de conteúdo: projeção t-SNE do texto (questão 5)

**Por que este gráfico.** As disciplinas têm texto livre (nome, objetivos e conteúdo), que não cabe em nenhum dos gráficos anteriores. Para responder se a estrutura institucional reflete o conteúdo, vetorizamos o texto com TF-IDF, reduzimos a dimensionalidade com SVD e projetamos em 2D com t-SNE. Cada ponto é uma disciplina, e pontos próximos têm vocabulário parecido. Na legenda, um clique esconde ou mostra um grupo, e um duplo clique isola o grupo. O menu alterna a cor entre grande área e campus. O tooltip mostra o nome da disciplina e o programa.

Os eixos do t-SNE não têm unidade, e só a vizinhança importa. Por isso eles ficam ocultos.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# Texto de cada disciplina: nome (duas vezes, para pesar mais), objetivos e conteúdo
texto = (df['disciplina'] + ' ' + df['disciplina'] + ' ' + df['objetivos'].fillna('') + ' ' + df['conteudo'].fillna('')).str.lower()

# TF-IDF: palavras com 3+ letras, ignorando termos raros (< 3 disciplinas) e muito comuns (> 40%)
vetorizador = TfidfVectorizer(min_df=3, max_df=0.4, sublinear_tf=True, token_pattern=r'(?u)\b[^\W\d_]{3,}\b')
X = vetorizador.fit_transform(texto)
print(f'Matriz TF-IDF: {X.shape[0]} disciplinas x {X.shape[1]} termos')

# SVD para 100 dimensões e normalização (distância euclidiana vira equivalente à de cosseno)
Z = normalize(TruncatedSVD(n_components=100, random_state=0).fit_transform(X))

# Projeção 2D
emb = TSNE(n_components=2, perplexity=30, init='pca', random_state=0).fit_transform(Z)
df['tsne_x'] = emb[:, 0]
df['tsne_y'] = emb[:, 1]
print('Projeção t-SNE concluída.')

Matriz TF-IDF: 2538 disciplinas x 10277 termos
Projeção t-SNE concluída.


In [15]:
ESQUEMAS = {
    'Grande área': ('grande_area', CORES_AREA),
    'Campus': ('campus', CORES_CAMPUS),
}

fig = go.Figure()
faixas = {}
for nome_esquema, (coluna, paleta) in ESQUEMAS.items():
    inicio = len(fig.data)
    for categoria, cor in paleta.items():
        sub = df[df[coluna] == categoria]
        fig.add_trace(go.Scattergl(
            x=sub['tsne_x'], y=sub['tsne_y'], mode='markers', name=categoria,
            marker=dict(size=6, color=cor, opacity=0.75),
            customdata=sub[['disciplina', 'nome_programa', 'grande_area', 'campus']].to_numpy(),
            hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]} · %{customdata[3]}<extra></extra>',
            visible=(nome_esquema == 'Grande área'),
        ))
    faixas[nome_esquema] = (inicio, len(fig.data))

botoes = []
for nome_esquema, (ini, fim) in faixas.items():
    visivel = [ini <= j < fim for j in range(len(fig.data))]
    botoes.append(dict(label=f'Cor: {nome_esquema}', method='update', args=[{'visible': visivel}]))

fig.update_layout(updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, xanchor='left',
                                    y=1.0, yanchor='bottom', showactive=True)],
                  legend=dict(itemsizing='constant'))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
ajustar(fig, 'Disciplinas projetadas por similaridade de texto (t-SNE sobre TF-IDF)', 700)
fig.show()

Só olhar a projeção pode enganar, porque o t-SNE distorce distâncias. Para ter um número que não depende dele, calculamos, para cada disciplina, quantos dos 10 vizinhos mais próximos no espaço TF-IDF/SVD (antes do t-SNE) são da mesma grande área, e comparamos com o esperado ao acaso, que é a fração de disciplinas daquela área no total.

In [16]:
# Vizinhos mais próximos no espaço vetorial (sem o t-SNE); o primeiro vizinho é a própria disciplina
vizinhos = NearestNeighbors(n_neighbors=11, metric='cosine').fit(Z)
_, idx = vizinhos.kneighbors(Z)
idx = idx[:, 1:]

def pureza(coluna):
    valores = df[coluna].to_numpy(dtype=object)
    return (valores[idx] == valores[:, None]).mean(axis=1)

df['pureza_area'] = pureza('grande_area')

tab = pd.DataFrame({
    'vizinhos da mesma área': df.groupby('grande_area')['pureza_area'].mean(),
    'esperado ao acaso': df['grande_area'].value_counts(normalize=True),
}).round(3)
tab['vezes acima do acaso'] = (tab['vizinhos da mesma área'] / tab['esperado ao acaso']).round(1)
display(tab)

print(f'Mesma grande área: {df["pureza_area"].mean():.1%} dos vizinhos '
      f'(ao acaso: {(df["grande_area"].value_counts(normalize=True) ** 2).sum():.1%})')
print(f'Mesmo campus: {pureza("campus").mean():.1%} '
      f'(ao acaso: {(df["campus"].value_counts(normalize=True) ** 2).sum():.1%})')
print(f'Mesmo programa: {pureza("nome_programa").mean():.1%}')

# Efeito do idioma: disciplinas fora do português têm vizinhos também fora do português?
fora_pt = (df['idioma_cat'] != 'Português').to_numpy()
print(f'Vizinhos fora do português, entre disciplinas fora do português: {fora_pt[idx][fora_pt].mean():.1%} '
      f'(proporção geral: {fora_pt.mean():.1%})')

# Disciplinas cujos vizinhos não são de sua área (candidatas a "pontes")
display(df.sort_values('pureza_area')[['disciplina', 'nome_programa', 'grande_area', 'pureza_area']].head(10))

,vizinhos da mesma área,esperado ao acaso,vezes acima do acaso
grande_area,,,
Ciências Agrárias,0.460,0.102,4.5
Ciências Biológicas e da Saúde,0.678,0.268,2.5
Ciências Exatas e da Terra,0.420,0.123,3.4
Engenharias,0.611,0.168,3.6
Humanidades e Sociais Aplicadas,0.729,0.270,2.7
Multidisciplinar,0.170,0.069,2.5


Mesma grande área: 59.1% dos vizinhos (ao acaso: 20.3%)
Mesmo campus: 50.5% (ao acaso: 36.5%)
Mesmo programa: 20.5%
Vizinhos fora do português, entre disciplinas fora do português: 14.0% (proporção geral: 6.2%)


,disciplina,nome_programa,grande_area,pureza_area
1138,Fundamentos de Análise de Dados,Administração,Humanidades e Sociais Aplicadas,0.0
14,Palavras Fundamentais em Ciência: Reflexões pa...,Instituto de Biociências,Ciências Biológicas e da Saúde,0.0
1787,Interação da Radiação Óptica com Tecidos Bioló...,Física Aplicada à Medicina e Biologia,Multidisciplinar,0.0
1147,Análise de Dados de Microbioma,Interunidades Sistemas Integrados em Alimentos,Multidisciplinar,0.0
1802,Seminários em Computação Aplicada a Biociências I,Computação Aplicada,Multidisciplinar,0.0
1805,Engenharia de Software,Computação Aplicada,Multidisciplinar,0.0
2472,Soluções Aplicadas para Segurança Hídrica,Engenharia Hidráulica e Saneamento,Engenharias,0.0
2460,Fundamentos de Mecânica Quântica para Engenhar...,Engenharia Elétrica,Engenharias,0.0
69,Características Básicas da Madeira II,Recursos Florestais,Ciências Agrárias,0.0
1050,Tecnologias de Biorrefinarias e Energia de Bio...,Engenharia de Sistemas Agrícolas,Ciências Agrárias,0.0


**O que vemos.** O texto tem relação clara com a área: em média, 59% dos 10 vizinhos mais próximos de uma disciplina são da mesma grande área, contra cerca de 20% ao acaso. A relação varia por área. Humanas e Sociais (73%) e Biológicas e Saúde (68%) são as mais coesas, seguidas de Engenharias (61%), e Agrárias (46%) e Exatas (42%) ficam mais abertas. Multidisciplinar tem só 17% de vizinhos da própria área, mas isso é 2,5 vezes o acaso (7%), e na projeção aparece espalhada entre as outras áreas, o que combina com o nome da categoria.

Já o programa explica pouco: só 20% dos vizinhos mais próximos são do mesmo programa, então disciplinas de programas diferentes se parecem entre si mais do que disciplinas de um mesmo programa. Com o campus a proporção é de cerca de 50% (ao acaso, 37%), mas boa parte disso é herdada da área, pela dependência entre campus e área vista no heatmap.

Entre as disciplinas com poucos vizinhos da própria área há nomes curtos ou genéricos ("Seminários de Geofísica I", "Ciência de Alimentos") e disciplinas que misturam campos já no título, como "Análise por Elementos Finitos Aplicada à Odontologia" e "Métodos Estatísticos Aplicados às Ciências Biológicas". Esses casos são bons candidatos para a análise da segunda etapa.

O idioma também interfere: entre as disciplinas fora do português, 14% dos vizinhos também estão fora do português (no conjunto todo são 6,2%), então parte da proximidade vem da língua e não do tema.

## Discussões preliminares

### Respondendo às questões

**Questão 1 (perfil de carga).** A carga semanal total é parecida entre as áreas (mediana de 8 a 10 horas), mas a composição muda bastante. Engenharias e Exatas dependem mais de estudo autônomo (mediana de 7 h e 4 h por semana, contra 2 h em Biológicas e Saúde e em Humanas e Sociais) e têm pouca prática. Biológicas e Saúde é a área mais prática. Ou seja, um número de créditos igual esconde formas diferentes de trabalhar.

**Questão 2 (estrutura da oferta).** A oferta é concentrada: São Paulo tem 56% das disciplinas e 64% dos programas, Ribeirão Preto vem em segundo, e 10 programas (4% do total) respondem por 21% das disciplinas.

**Questão 3 (formato).** Disciplinas intensivas e não presenciais não estão distribuídas por igual. Intensivas se concentram em Biológicas e Saúde, Agrárias e Multidisciplinar, e as não presenciais são mais frequentes no Intercampi, em Pirassununga e em Bauru. Disciplinas fora do português são poucas e se concentram em Exatas.

**Questão 4 (casos extremos).** Os maiores valores de carga semanal vêm de disciplinas de 1 ou 2 semanas, e são um efeito da duração. A razão estudo/aula alta é outro tipo de caso, em que há muito estudo para poucas horas de aula, e aparece em um grupo pequeno de disciplinas (166, ou 6,5%).

**Questão 5 (similaridade de conteúdo).** O texto concorda com a área de forma clara (59% de vizinhos da mesma área, contra 20% ao acaso), mas concorda muito menos com o programa (20%). Multidisciplinar não forma um grupo próprio no texto, e algumas disciplinas ficam entre áreas.

### Interpretação crítica e limitações
- **Unidade de análise.** Cada linha é uma disciplina cadastrada no Janus, o que não significa que tenha sido oferecida em turmas. Um programa grande no treemap tem muito cadastro, mas isso não diz nada sobre o número de alunos nem sobre a frequência com que as disciplinas abrem.
- **Campus e área não são independentes.** Os dois atributos vêm do mapeamento comissão para campus e área que fizemos à mão. A área, em particular, é uma classificação nossa, e outra pessoa poderia agrupar de outro jeito (por exemplo, onde colocar a Educação Física ou a Música). Várias comparações entre campi são, na prática, comparações entre áreas.
- **Células pequenas.** No heatmap, percentuais de células com poucas disciplinas variam muito. O tooltip mostra o `n`, mas é fácil esquecer de olhar.
- **t-SNE e TF-IDF.** O t-SNE não preserva distâncias globais e muda com a semente e a perplexidade, e o TF-IDF não entende significado (só vocabulário). Disciplinas em inglês tendem a ficar próximas entre si pelo idioma, não pelo tema, e textos repetidos entre disciplinas (há conteúdos iguais em até 5 registros) podem aproximar disciplinas sem relação. Por isso fizemos o teste dos vizinhos, que não depende da projeção, mas ele também usa o mesmo vocabulário.
- **Dados de docentes.** 40% das disciplinas não têm docente informado, por isso deixamos esse campo de fora desta etapa.